In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
HuggingFaceApi = os.getenv('HF_TOKEN')
Groq_Api_Key = os.getenv('GROQ_API_KEY')

In [2]:
import random
import pandas as pd
import numpy as np

def Create_df(n):
    name = 'Sonu Monu Nishu Rahul Nishita Punam Mukesh'.split()
    NameCols = [random.choice(name) for _ in range(n+1)]
    Salary  = [np.random.randint(100) for _ in range(n+1)]
    df = pd.DataFrame(list(zip(NameCols, Salary)), columns = ['Name', 'Salary'])
    return df

In [3]:
df = Create_df(10)
df.sample(2)

,Name,Salary
1,Mukesh,31
3,Rahul,16


In [6]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
engine = create_engine("sqlite:///random_dataframe.db")
df.to_sql('random_dataframe', engine, index=False)
db = SQLDatabase(engine=engine)

In [7]:
db.get_usable_table_names
print(db.run("Select * From random_dataframe"))

[('Monu', 46), ('Mukesh', 31), ('Punam', 30), ('Rahul', 16), ('Nishita', 44), ('Rahul', 83), ('Mukesh', 52), ('Punam', 74), ('Punam', 81), ('Rahul', 33), ('Monu', 48)]


In [8]:
from langchain_groq import ChatGroq
llm = ChatGroq(model = 'llama3-8b-8192', api_key=Groq_Api_Key,
               temperature= 0.9 , max_tokens=1024)

In [9]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()
tools

[QuerySQLDataBaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000244D3B7D450>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000244D3B7D450>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000244D3B7D450>),
 QuerySQLCheckerTool(description='Use this tool to 

In [10]:
from langchain import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

# assert len(prompt_template.messages) == 1
# prompt_template.messages[0].pretty_print()


c:\vs_code\Genai\genai_env\Lib\site-packages\langsmith\client.py:5499: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


In [11]:
system_message = prompt_template.format(dialect="SQLite", top_k=5)

In [12]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools, state_modifier=system_message)

In [15]:
db.run(''' 
        SELECT SUM(Salary) As Total_Salary FROM random_dataframe
       WHERE Name = 'Monu'
        GROUP BY Name ''')

'[(94,)]'

In [18]:
df[df['Name']=='Monu']

,Name,Salary
0,Monu,46
10,Monu,48


In [23]:
question = "what is the total count of the name Monu"

for step in agent_executor.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

what is the total count of the name Monu
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_pnfk)
 Call ID: call_pnfk
  Args:
    query: SELECT COUNT(*) FROM table_name WHERE name = 'Monu'
================================= Tool Message =================================
Name: sql_db_query

Error: (sqlite3.OperationalError) no such table: table_name
[SQL: SELECT COUNT(*) FROM table_name WHERE name = 'Monu']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_rmrn)
 Call ID: call_rmrn
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

random_dataframe
================================== Ai Message ==================================
Tool Calls:
  sql_db_schem